# Scene batch

klein draws one complete vertical scene per job, conditioned on costume reference pictures.
Every number in the report is measured; a failed picture is recorded, not hidden.


In [ ]:
import os, sys, json, time, subprocess, pathlib, hashlib, tarfile, traceback, uuid, urllib.request

RUN_ID = globals().get("RUN_ID", "manual-" + uuid.uuid4().hex)
RUN_TOKEN = globals().get("RUN_TOKEN", "manual")
COMMIT_SHA = globals().get("COMMIT_SHA", "unknown")
WORKFLOW_RUN_ID = globals().get("WORKFLOW_RUN_ID", "local")

OUT = pathlib.Path(os.environ.get("STICKMAN_OUT_DIR", "/kaggle/working/outputs"))  # override only for unit tests
WORK = pathlib.Path(os.environ.get("STICKMAN_WORK_DIR", "/tmp/scene_batch"))
LOGS = OUT / "logs"
for d in (OUT, LOGS, WORK):
    d.mkdir(parents=True, exist_ok=True)

REPO = "damarlanandu2005-jpg/stickman-motion-studio"
FALLBACK_REF = "test/image-to-3d-objects"
REFDIR = "animation/reference_art/"
REFS = {
 "costumes/costume_doctor.png": "28878fad2564e161cb6eeffcc6eac4f4f8bb60c386f50966526497afd7790d94",
 "costumes/costume_explorer.png": "436dbfe14738f5c2eab9165d2c6594cc6d41c4eee85a72c1a52fc81b01a5ea62",
 "costumes/costume_sailor.png": "2a81a663c678b3c7c5b75005b8a9ebc37842e1a94482f8372b99acaa1aeda8d8",
 "costumes/costume_soldier.png": "43b598d9ca43b9d422cef25d45f64860de6e0d1e9254c3ec87f3825c292c362e"
}

KLEIN_ID, KLEIN_REVISION = "black-forest-labs/FLUX.2-klein-4B", "e7b7dc27f91deacad38e78976d1f2b499d76a294"
KLEIN_PIP = ["diffusers==0.42.1", "transformers==5.22.1", "huggingface-hub==1.33.0", "accelerate", "pillow"]
CACHE_ENV = {"HF_HOME": "/tmp/hf_home", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1"}

JOBS = {
 "c01": {
  "refs": [
   "costumes/costume_soldier.png"
  ],
  "prompt": "Medium shot: the characters are shown from the chest up and fill the lower half of the frame; the background is visible above and behind them. Characters: a stickman in the costume of reference image 1, wearing a thick green bomb disposal suit with a round helmet. Keep each costume exactly as in its reference image and the same big round white head. Place: grey street barrier with yellow tape. Action: kneels over a box of coloured wires with a timer showing 0:03. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6100,
  "width": 576,
  "height": 1024
 },
 "c02": {
  "refs": [
   "costumes/costume_soldier.png"
  ],
  "prompt": "Medium shot: the characters are shown from the chest up and fill the lower half of the frame; the background is visible above and behind them. Characters: a stickman in the costume of reference image 1, wearing a thick green bomb disposal suit with a round helmet. Keep each costume exactly as in its reference image and the same big round white head. Place: grey smoky street. Action: stands fading in smoke. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6101,
  "width": 576,
  "height": 1024
 },
 "c03": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, dressed as a police officer in a dark navy uniform and cap; a stickman in the costume of reference image 1, police officer wearing a headset with a microphone; a stickman in the costume of reference image 1, dressed as a police officer in a dark navy uniform and cap. Keep each costume exactly as in its reference image and the same big round white head. Place: police garage. Action: three officers stand in a row, one with a moustache. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6102,
  "width": 576,
  "height": 1024
 },
 "c04": {
  "refs": [
   "costumes/costume_explorer.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, wearing a mustard yellow coat and no hat, brown hair. Keep each costume exactly as in its reference image and the same big round white head. Place: city street corner with a red traffic light and shops. Action: stands looking up. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6103,
  "width": 576,
  "height": 1024
 },
 "c05": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: plain grey background. Objects: a big black traffic light with all lights off. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6104,
  "width": 576,
  "height": 1024
 },
 "c06": {
  "refs": [
   "costumes/costume_doctor.png",
   "costumes/costume_explorer.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, an old doctor with glasses and a white coat; a stickman in the costume of reference image 2, wearing a mustard yellow coat and no hat, brown hair. Keep each costume exactly as in its reference image and the same big round white head. Place: bright clinic room. Action: the doctor holds a red heart card up to the man. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6105,
  "width": 576,
  "height": 1024
 },
 "c07": {
  "refs": [
   "costumes/costume_explorer.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, wearing a mustard yellow coat and no hat, brown hair. Keep each costume exactly as in its reference image and the same big round white head. Place: police station wall with a notice board and notes. Action: points at the notice board. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6106,
  "width": 576,
  "height": 1024
 },
 "c08": {
  "refs": [],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Place: front of a grey police department building with columns and steps and a sign. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6107,
  "width": 576,
  "height": 1024
 },
 "c09": {
  "refs": [
   "costumes/costume_soldier.png"
  ],
  "prompt": "Extreme close-up: one character's big round head and face fill most of the picture, strong clear expression, only a hint of the costume at the bottom edge. Characters: a stickman in the costume of reference image 1, wearing a thick green bomb disposal suit with a round helmet. Keep each costume exactly as in its reference image and the same big round white head. Place: grey background. Action: big round helmet face with wide eyes behind the visor. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6108,
  "width": 576,
  "height": 1024
 },
 "c10": {
  "refs": [
   "costumes/costume_doctor.png",
   "costumes/costume_explorer.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, an old doctor with glasses and a white coat; a stickman in the costume of reference image 2, wearing a mustard yellow coat and no hat, brown hair. Keep each costume exactly as in its reference image and the same big round white head. Place: clinic with a wall chart and a clock. Action: the doctor holds up a chart, the man looks at it. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6109,
  "width": 576,
  "height": 1024
 },
 "c11": {
  "refs": [
   "costumes/costume_explorer.png"
  ],
  "prompt": "Medium shot: the characters are shown from the chest up and fill the lower half of the frame; the background is visible above and behind them. Characters: a stickman in the costume of reference image 1, wearing a mustard yellow coat and no hat, brown hair. Keep each costume exactly as in its reference image and the same big round white head. Place: dark night room with a desk lamp and a window with a moon. Action: sits at a desk studying. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6110,
  "width": 576,
  "height": 1024
 },
 "c12": {
  "refs": [],
  "prompt": "A flat plain colour background with one big simple object in the centre. No characters and no scenery. Place: plain orange background. Objects: a big round white clock face showing 12. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6111,
  "width": 576,
  "height": 1024
 },
 "c13": {
  "refs": [
   "costumes/costume_doctor.png",
   "costumes/costume_explorer.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, an old doctor with glasses and a white coat; a stickman in the costume of reference image 2, wearing a mustard yellow coat and no hat, brown hair. Keep each costume exactly as in its reference image and the same big round white head. Place: bright clinic room. Action: they walk together. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6112,
  "width": 576,
  "height": 1024
 },
 "c14": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, dressed as a police officer in a dark navy uniform and cap; a stickman in the costume of reference image 1, police officer wearing a headset with a microphone; a stickman in the costume of reference image 1, dressed as a police officer in a dark navy uniform and cap. Keep each costume exactly as in its reference image and the same big round white head. Place: garage with grey roller doors. Action: three officers raise their hands and cheer. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6113,
  "width": 576,
  "height": 1024
 },
 "c15": {
  "refs": [],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Place: city street with a blue armoured police truck and shops. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6114,
  "width": 576,
  "height": 1024
 },
 "c16": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Medium shot: the characters are shown from the chest up and fill the lower half of the frame; the background is visible above and behind them. Characters: a stickman in the costume of reference image 1, police officer wearing a headset with a microphone. Keep each costume exactly as in its reference image and the same big round white head. Place: garage wall. Action: stands thinking with a thought bubble holding a green check mark. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6115,
  "width": 576,
  "height": 1024
 },
 "c17": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, police officer with a headset and moustache. Keep each costume exactly as in its reference image and the same big round white head. Place: control room with a city map and monitors. Action: points at the map. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6116,
  "width": 576,
  "height": 1024
 },
 "c18": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: dark grey bomb casing. Objects: a digital timer showing 0:30 above red green yellow and blue wires. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6117,
  "width": 576,
  "height": 1024
 },
 "c19": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Extreme close-up: one character's big round head and face fill most of the picture, strong clear expression, only a hint of the costume at the bottom edge. Characters: a stickman in the costume of reference image 1, police officer wearing a headset with a microphone. Keep each costume exactly as in its reference image and the same big round white head. Place: grey background. Action: big worried face with a sweat drop, headset microphone. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6118,
  "width": 576,
  "height": 1024
 },
 "c20": {
  "refs": [
   "costumes/costume_explorer.png"
  ],
  "prompt": "Medium shot: the characters are shown from the chest up and fill the lower half of the frame; the background is visible above and behind them. Characters: a stickman in the costume of reference image 1, wearing a mustard yellow coat and no hat, brown hair. Keep each costume exactly as in its reference image and the same big round white head. Place: dark night room with a desk lamp, moon in the window and a clock. Action: sits asleep over a book. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6119,
  "width": 576,
  "height": 1024
 },
 "c21": {
  "refs": [],
  "prompt": "A flat plain colour background with one big simple object in the centre. No characters and no scenery. Place: orange background. Objects: three blue blueprint sheets with simple technical diagrams of a factory. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6120,
  "width": 576,
  "height": 1024
 },
 "c22": {
  "refs": [],
  "prompt": "A flat plain colour background with one big simple object in the centre. No characters and no scenery. Place: plain blue background. Objects: a blueprint drawing of four batteries in a row. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6121,
  "width": 576,
  "height": 1024
 },
 "c23": {
  "refs": [
   "costumes/costume_explorer.png"
  ],
  "prompt": "Medium shot: the characters are shown from the chest up and fill the lower half of the frame; the background is visible above and behind them. Characters: a stickman in the costume of reference image 1, wearing a mustard yellow coat and no hat, brown hair. Keep each costume exactly as in its reference image and the same big round white head. Place: night desk with a tall stack of colourful books. Action: sits relaxed with half-closed eyes. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6122,
  "width": 576,
  "height": 1024
 },
 "c24": {
  "refs": [],
  "prompt": "A flat plain colour background with one big simple object in the centre. No characters and no scenery. Place: plain blue background. Objects: a blueprint circuit board drawing with a hand pointing at a terminal. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6123,
  "width": 576,
  "height": 1024
 },
 "c25": {
  "refs": [],
  "prompt": "A flat plain colour background with one big simple object in the centre. No characters and no scenery. Place: plain orange background. Objects: four coloured paper strips red green yellow blue fanned out. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6124,
  "width": 576,
  "height": 1024
 },
 "c26": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Medium shot: the characters are shown from the chest up and fill the lower half of the frame; the background is visible above and behind them. Characters: a stickman in the costume of reference image 1, police officer wearing a headset with a microphone. Keep each costume exactly as in its reference image and the same big round white head. Place: garage wall with a row of green check marks and a big red number 23. Action: stands with arms crossed. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6125,
  "width": 576,
  "height": 1024
 },
 "c27": {
  "refs": [],
  "prompt": "A flat plain colour background with one big simple object in the centre. No characters and no scenery. Place: plain beige background. Objects: a tear-off wall calendar showing YEAR 2. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6126,
  "width": 576,
  "height": 1024
 },
 "c28": {
  "refs": [
   "costumes/costume_sailor.png",
   "costumes/costume_doctor.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, dressed as a police officer in a dark navy uniform and cap; a stickman in the costume of reference image 2, an old doctor with glasses and a white coat. Keep each costume exactly as in its reference image and the same big round white head. Place: dark warehouse with a spotlight and a box of coloured wires. Action: they look at a device on the floor. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6127,
  "width": 576,
  "height": 1024
 },
 "c29": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: grey background. Objects: a bundle of grey twisted wires. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6128,
  "width": 576,
  "height": 1024
 },
 "c30": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Extreme close-up: one character's big round head and face fill most of the picture, strong clear expression, only a hint of the costume at the bottom edge. Characters: a stickman in the costume of reference image 1, police chief with a white shirt, cap and moustache, holding a radio to his mouth. Keep each costume exactly as in its reference image and the same big round white head. Place: police street scene with yellow tape. Action: shouting into the radio. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6129,
  "width": 576,
  "height": 1024
 },
 "c31": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: dark background. Objects: a timer showing 0:08 above a cut bundle of red blue yellow wires with sparks. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6130,
  "width": 576,
  "height": 1024
 },
 "c32": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Medium shot: the characters are shown from the chest up and fill the lower half of the frame; the background is visible above and behind them. Characters: a stickman in the costume of reference image 1, police officer wearing a headset with a microphone. Keep each costume exactly as in its reference image and the same big round white head. Place: dark warehouse with a spotlight. Action: holds both hands on his headset. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6131,
  "width": 576,
  "height": 1024
 },
 "c33": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Extreme close-up: one character's big round head and face fill most of the picture, strong clear expression, only a hint of the costume at the bottom edge. Characters: a stickman in the costume of reference image 1, police officer wearing a headset with a microphone. Keep each costume exactly as in its reference image and the same big round white head. Place: dark background. Action: huge calm face with eyes nearly closed. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6132,
  "width": 576,
  "height": 1024
 },
 "c34": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: pale grey background. Objects: a grey bundle of wires with a timer showing 0:05. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6133,
  "width": 576,
  "height": 1024
 },
 "c35": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: pale grey background. Objects: a tangled knot of grey wires with a timer showing 0:03 and a red wire in a clamp. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6134,
  "width": 576,
  "height": 1024
 },
 "c36": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: dark crate in a warehouse. Objects: a digital timer showing 0:01 above red blue yellow green wires on a wooden crate. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6135,
  "width": 576,
  "height": 1024
 },
 "c37": {
  "refs": [
   "costumes/costume_sailor.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, police chief in white shirt and cap with moustache; a stickman in the costume of reference image 1, police officer wearing a headset with a microphone. Keep each costume exactly as in its reference image and the same big round white head. Place: garage with grey roller doors. Action: the chief talks, the other officer listens with a speech bubble showing the number 74. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6136,
  "width": 576,
  "height": 1024
 },
 "c38": {
  "refs": [
   "costumes/costume_doctor.png",
   "costumes/costume_sailor.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, head instructor; a stickman in the costume of reference image 2, police officer wearing a headset with a microphone; a stickman in the costume of reference image 2, dressed as a police officer in a dark navy uniform and cap. Keep each costume exactly as in its reference image and the same big round white head. Place: classroom with a wooden sign on the wall and desks. Action: the instructor points at the students. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6137,
  "width": 576,
  "height": 1024
 },
 "c39": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: city skyline at sunset over water, orange and purple sky. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6138,
  "width": 576,
  "height": 1024
 },
 "c40": {
  "refs": [
   "costumes/costume_sailor.png",
   "costumes/costume_soldier.png"
  ],
  "prompt": "Wide shot: the characters are small in the frame, about one fifth of the picture height, standing on the ground in the lower third; a rich detailed environment fills the rest of the picture. Characters: a stickman in the costume of reference image 1, police officer wearing a headset with a microphone; a stickman in the costume of reference image 2, wearing a thick green bomb disposal suit with a round helmet. Keep each costume exactly as in its reference image and the same big round white head. Place: classroom with a green board. Action: one is holding a paint can, both smiling. Flat 2D vector cartoon in the exact style of the reference stickman character: big round white head with a thick dark outline, small oval dot eyes, simple eyebrows and mouth, thin clean outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration with a coherent detailed background. No text, no letters, no captions, no watermark, no panel borders, no collage.",
  "seed": 6139,
  "width": 576,
  "height": 1024
 },
 "a24": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: pale snow background. Objects: a wooden post with a coin flying above it and small sparkles. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6140,
  "width": 576,
  "height": 1024
 },
 "b10": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: night sky with stars. Objects: a big military cargo plane in the distance with an orange explosion flash. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6141,
  "width": 576,
  "height": 1024
 },
 "b11": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: night sky. Objects: a giant orange fire explosion with dark smoke over a black ridge and a grey rock in front. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6142,
  "width": 576,
  "height": 1024
 },
 "b12": {
  "refs": [],
  "prompt": "Close-up of the object only. No characters in the picture. Place: night sky. Objects: a bright yellow and orange explosion with black smoke over a grey rock. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no captions, no watermark, no panel borders.",
  "seed": 6143,
  "width": 576,
  "height": 1024
 }
}
STEPS, GUIDANCE = 4, 1.0

REPORT = {
    "run_token": RUN_TOKEN, "run_id": RUN_ID, "commit_sha": COMMIT_SHA, "workflow_run_id": WORKFLOW_RUN_ID,
    "status": "running", "error_message": None, "poses": list(JOBS), "seeds": [0], "reference": REFDIR,
    "environment": {}, "klein": {}, "files": [],
}
STARTED = time.monotonic()


In [ ]:
KLEIN_STAGE = 'import json, sys, time, hashlib, traceback\nfrom pathlib import Path\ncfg = json.loads(Path(sys.argv[1]).read_text())\nres = {"stage": "klein", "status": "running", "error": None, "pictures": []}\ntry:\n    import torch\n    from PIL import Image\n    from huggingface_hub import snapshot_download\n    from diffusers import Flux2KleinPipeline\n    t = time.monotonic(); path = snapshot_download(cfg["model_id"], revision=cfg["revision"])\n    res["download_seconds"] = round(time.monotonic() - t, 1)\n    t = time.monotonic(); pipe = Flux2KleinPipeline.from_pretrained(path, torch_dtype=torch.float16)\n    pipe.enable_model_cpu_offload(gpu_id=0); res["load_seconds"] = round(time.monotonic() - t, 1)\n    out = Path(cfg["out_dir"]); out.mkdir(parents=True, exist_ok=True)\n    for name, job in cfg["jobs"].items():\n        item = {"pose": name, "seed": job["seed"], "status": "running"}\n        res["pictures"].append(item)\n        try:\n            refs = []\n            for r in job["refs"]:\n                im = Image.open(Path(cfg["refdir"]) / r).convert("RGB"); im.thumbnail((1024, 1024)); refs.append(im)\n            torch.cuda.synchronize(); t = time.monotonic()\n            kw = {"image": refs} if refs else {}  # object-only pictures use no reference, so no mascot face leaks onto objects\n            img = pipe(prompt=job["prompt"], width=job["width"], height=job["height"], num_inference_steps=cfg["steps"],\n                       guidance_scale=cfg["guidance"], generator=torch.Generator("cpu").manual_seed(job["seed"]), **kw).images[0]\n            torch.cuda.synchronize(); secs = round(time.monotonic() - t, 2)\n            p = out / f"pose_{name}.png"; img.save(p)\n            item.update({"file": p.name, "seconds": secs, "width": img.width, "height": img.height,\n                         "sha256": hashlib.sha256(p.read_bytes()).hexdigest(), "status": "success"})\n            print("klein drew", p.name, "in", secs, "s", flush=True)\n        except Exception as exc:  # one bad picture must not stop the rest\n            item.update({"status": "failed", "error": f"{type(exc).__name__}: {exc}"[:500]}); traceback.print_exc()\n            try:\n                torch.cuda.empty_cache()\n            except Exception:\n                pass\n    res["peak_vram_gib"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)\n    res["status"] = "success" if all(i["status"] == "success" for i in res["pictures"]) else "partial"\nexcept Exception as exc:\n    res["status"] = "failed"; res["error"] = f"{type(exc).__name__}: {exc}"[:3000]; traceback.print_exc()\nPath(cfg["result"]).write_text(json.dumps(res, indent=2))\n'


In [ ]:
def run_logged(name, cmd, timeout):
    log = LOGS / f"{name}.log"
    t = time.monotonic()
    try:
        with log.open("a") as fh:
            fh.write("$ " + " ".join(map(str, cmd)) + "\n"); fh.flush()
            code = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, timeout=timeout, env={**os.environ, **CACHE_ENV}).returncode
        outcome = "ok" if code == 0 else f"exit {code}"
    except subprocess.TimeoutExpired:
        code, outcome = None, f"timed out after {timeout} s"
    print(f"[{name}] {outcome} in {round(time.monotonic() - t, 1)} s", flush=True)
    return code == 0


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def fetch_reference():
    ref = COMMIT_SHA if len(COMMIT_SHA) == 40 else FALLBACK_REF
    tgz = WORK / "repo.tar.gz"
    urllib.request.urlretrieve(f"https://codeload.github.com/{REPO}/tar.gz/{ref}", tgz)
    with tarfile.open(tgz) as tf:
        tf.extractall(WORK / "repo", filter="data")
    folder = next((WORK / "repo").iterdir()) / REFDIR
    for name, digest in REFS.items():
        if sha256_of(folder / name) != digest:
            raise RuntimeError(f"reference picture {name} does not match the approved file (hash differs)")
    return folder


def environment():
    import torch
    env = REPORT["environment"]
    env["python"] = sys.version.split()[0]
    env["torch"] = torch.__version__
    env["cuda_available"] = bool(torch.cuda.is_available())
    env["gpu_names"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []
    if not env["cuda_available"]:
        raise RuntimeError("CUDA is not available: enable the GPU accelerator.")


In [ ]:
try:
    environment()
    reference = fetch_reference()
    if not run_logged("install_klein", [sys.executable, "-m", "pip", "install", "--no-cache-dir", *KLEIN_PIP], 1200):
        raise RuntimeError("klein dependencies failed to install; see logs/install_klein.log")
    script, cfg, result = WORK / "klein_stage.py", WORK / "klein_stage.json", OUT / "result_klein.json"
    script.write_text(KLEIN_STAGE, encoding="utf-8")
    cfg.write_text(json.dumps({"model_id": KLEIN_ID, "revision": KLEIN_REVISION, "refdir": str(reference), "jobs": JOBS,
                               "steps": STEPS, "guidance": GUIDANCE,
                               "result": str(result), "out_dir": str(OUT)}), encoding="utf-8")
    run_logged("klein", [sys.executable, str(script), str(cfg)], 3000)
    if not result.exists():
        raise RuntimeError("klein wrote no result; see logs/klein.log")
    REPORT["klein"] = json.loads(result.read_text(encoding="utf-8"))
    REPORT["files"] = [p["file"] for p in REPORT["klein"]["pictures"] if p.get("status") == "success"]
    if REPORT["klein"]["status"] == "failed":
        raise RuntimeError("klein failed: " + str(REPORT["klein"].get("error")))
    REPORT["status"] = "completed"  # per-picture success is in REPORT["klein"]["pictures"]
except Exception as exc:
    REPORT["status"] = "failed"
    REPORT["error_message"] = f"{type(exc).__name__}: {exc}"
    traceback.print_exc()
finally:
    REPORT["notebook_duration_seconds"] = round(time.monotonic() - STARTED, 1)
    (OUT / "poses_report.json").write_text(json.dumps(REPORT, indent=2), encoding="utf-8")
    print(json.dumps({k: REPORT[k] for k in ("status", "error_message", "files", "notebook_duration_seconds")}, indent=2))


In [ ]:
assert REPORT["status"] == "completed", "Poses did not complete: " + str(REPORT["error_message"])
print("Drew", len(REPORT["files"]), "of", len(JOBS), "pictures")
